# 🥈 Silver Layer Architecture

The Silver layer represents the curated enterprise operational layer of the Lakehouse architecture.

This layer transforms raw Bronze operational datasets into:

* cleansed datasets
* enriched datasets
* standardized business entities
* AI-ready operational data
* retrieval-optimized structures

The Silver layer serves as the foundational data layer for:

* advanced analytics
* enterprise AI
* retrieval pipelines
* embeddings
* graph intelligence
* Agentic AI workflows

Key objectives of the Silver layer include:

* data quality improvement
* semantic enrichment
* denormalization
* metadata engineering
* retrieval optimization
* scalable downstream consumption

Unlike Bronze datasets, Silver datasets are intentionally designed for enterprise intelligence and AI consumption.


# ⚙️ Environment & Configuration Initialization

In [0]:
# ==========================================
# Environment & Configuration Initialization
# ==========================================

# Core Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Enterprise Configuration
CONFIG = {

    # Catalog & Schema
    "catalog": "spark_catalog",
    "schema": "uber_ai",

    # Partitioning
    "partition_column": "event_date",

    # Environment Metadata
    "environment": "dev"
}

print("✅ Environment Initialized")
print(CONFIG)

# ⚙️ Silver Layer Configuration

In [0]:
# ==========================================
# Silver Layer Configuration
# ==========================================

SILVER_TABLES = {

    # Geographic Intelligence
    "cities_silver":
        f"{CONFIG['schema']}.cities_silver",

    # Surge Intelligence
    "pricing_zones_silver":
        f"{CONFIG['schema']}.pricing_zones_silver",

    # Driver Intelligence
    "drivers_silver":
        f"{CONFIG['schema']}.drivers_silver",

    # Customer Intelligence
    "riders_silver":
        f"{CONFIG['schema']}.riders_silver",

    # Operational Intelligence
    "rides_silver":
        f"{CONFIG['schema']}.rides_silver",

    # Event Intelligence
    "trip_events_silver":
        f"{CONFIG['schema']}.trip_events_silver",

    # Retrieval Intelligence
    "operational_documents":
        f"{CONFIG['schema']}.operational_documents"
}

print("✅ Silver Table Configuration Initialized")

for table_name, table_path in SILVER_TABLES.items():
    print(f"{table_name} --> {table_path}")

# 🥉 Read Bronze Layer Tables

In [0]:
# ==========================================
# Read Bronze Layer Tables
# ==========================================

cities_bronze_df = spark.table(
    f"{CONFIG['schema']}.cities"
)

pricing_zones_bronze_df = spark.table(
    f"{CONFIG['schema']}.pricing_zones"
)

drivers_bronze_df = spark.table(
    f"{CONFIG['schema']}.drivers"
)

riders_bronze_df = spark.table(
    f"{CONFIG['schema']}.riders"
)

rides_bronze_df = spark.table(
    f"{CONFIG['schema']}.rides"
)

trip_events_bronze_df = spark.table(
    f"{CONFIG['schema']}.trip_events"
)

print("✅ Bronze Tables Loaded Successfully")

# ✅ Bronze Layer Validation

In [0]:
# ==========================================
# Bronze Layer Record Counts
# ==========================================

validation_results = [

    ("cities", cities_bronze_df.count()),
    ("pricing_zones", pricing_zones_bronze_df.count()),
    ("drivers", drivers_bronze_df.count()),
    ("riders", riders_bronze_df.count()),
    ("rides", rides_bronze_df.count()),
    ("trip_events", trip_events_bronze_df.count())
]

validation_df = spark.createDataFrame(
    validation_results,
    ["table_name", "record_count"]
)

display(validation_df)

# 🚘 Driver Silver Transformation

In [0]:
# ==========================================
# Driver Silver Transformation
# ==========================================

drivers_silver_df = (

    drivers_bronze_df

    # Standardize Driver Status
    .withColumn(
        "driver_status",
        upper(trim(col("driver_status")))
    )

    # Driver Active Flag
    .withColumn(
        "is_active",
        when(col("driver_status") == "ACTIVE", lit(1))
        .otherwise(lit(0))
    )

    # Experience Band
    .withColumn(
        "experience_band",
        when(col("years_experience") <= 2, "BEGINNER")
        .when(col("years_experience") <= 5, "INTERMEDIATE")
        .when(col("years_experience") <= 10, "EXPERIENCED")
        .otherwise("VETERAN")
    )

    # Rating Category
    .withColumn(
        "rating_category",
        when(col("driver_rating") >= 4.8, "ELITE")
        .when(col("driver_rating") >= 4.5, "HIGH")
        .when(col("driver_rating") >= 4.0, "MEDIUM")
        .otherwise("LOW")
    )

    # Metadata Columns
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )
)

print("✅ Driver Silver Transformation Completed")

# 👤 Rider Silver Transformation

In [0]:
# ==========================================
# Rider Silver Transformation
# ==========================================

riders_silver_df = (

    riders_bronze_df

    # Standardize Rider Status
    .withColumn(
        "rider_status",
        upper(trim(col("rider_status")))
    )

    # Rider Active Flag
    .withColumn(
        "is_active",
        when(col("rider_status") == "ACTIVE", lit(1))
        .otherwise(lit(0))
    )

    # Rider Loyalty Segment
    .withColumn(
        "rider_segment",
        when(col("rider_rating") >= 4.8, "PREMIUM")
        .when(col("rider_rating") >= 4.5, "LOYAL")
        .when(col("rider_rating") >= 4.0, "REGULAR")
        .otherwise("LOW_ENGAGEMENT")
    )

    # Email Domain
    .withColumn(
        "email_domain",
        split(col("email"), "@").getItem(1)
    )

    # Registration Year
    .withColumn(
        "registration_year",
        year(to_date(col("registration_date")))
    )

    # Metadata Columns
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )
)

print("✅ Rider Silver Transformation Completed")

# 🚖 Ride Silver Transformation

In [0]:
# ==========================================
# Ride Silver Transformation
# ==========================================

rides_silver_df = (

    rides_bronze_df.alias("r")

    # Join Riders
    .join(
        riders_silver_df.select(
            "rider_id",
            "rider_name",
            "rider_segment"
        ).alias("ri"),

        col("r.rider_id") == col("ri.rider_id"),
        "left"
    )

    # Join Drivers
    .join(
        drivers_silver_df.select(
            "driver_id",
            "driver_name",
            "vehicle_type",
            "rating_category"
        ).alias("d"),

        col("r.driver_id") == col("d.driver_id"),
        "left"
    )

    # Join Cities
    .join(
        cities_silver_df.select(
            "city_id",
            "city_name",
            "country",
            "region",
            "city_tier"
        ).alias("c"),

        col("r.city_id") == col("c.city_id"),
        "left"
    )

    # Join Pricing Zones
    .join(
        pricing_zones_silver_df.select(
            "zone_id",
            "zone_name",
            "surge_category"
        ).alias("z"),

        col("r.zone_id") == col("z.zone_id"),
        "left"
    )

    # Ride Value Category
    .withColumn(
        "ride_value_category",
        when(col("fare_amount") >= 2000, "PREMIUM")
        .when(col("fare_amount") >= 1000, "HIGH")
        .when(col("fare_amount") >= 500, "MEDIUM")
        .otherwise("LOW")
    )

    # Ride Distance Category
    .withColumn(
        "distance_category",
        when(col("distance_km") >= 30, "LONG_DISTANCE")
        .when(col("distance_km") >= 15, "MEDIUM_DISTANCE")
        .otherwise("SHORT_DISTANCE")
    )

    # Ride Time Category
    .withColumn(
        "ride_time_category",
        when(hour(col("ride_timestamp")).between(6, 11), "MORNING")
        .when(hour(col("ride_timestamp")).between(12, 17), "AFTERNOON")
        .when(hour(col("ride_timestamp")).between(18, 23), "EVENING")
        .otherwise("NIGHT")
    )

    # AI Retrieval Summary
    .withColumn(
        "ride_summary",
        concat_ws(
            " ",
            lit("Ride completed in"),
            col("c.city_name"),
            lit("by driver"),
            col("d.driver_name"),
            lit("for rider"),
            col("ri.rider_name"),
            lit("with fare amount"),
            col("fare_amount"),
            lit("and distance"),
            col("distance_km"),
            lit("kilometers.")
        )
    )

    # Metadata Columns
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )

    # Final Columns
    .select(

        col("r.ride_id"),

        col("r.rider_id"),
        col("ri.rider_name"),
        col("ri.rider_segment"),

        col("r.driver_id"),
        col("d.driver_name"),
        col("d.vehicle_type"),
        col("d.rating_category"),

        col("r.city_id"),
        col("c.city_name"),
        col("c.country"),
        col("c.region"),
        col("c.city_tier"),

        col("r.zone_id"),
        col("z.zone_name"),
        col("z.surge_category"),

        col("r.ride_timestamp"),
        col("r.ride_status"),

        col("r.fare_amount"),
        col("ride_value_category"),

        col("r.distance_km"),
        col("distance_category"),

        col("r.ride_duration_minutes"),
        col("ride_time_category"),

        col("r.payment_method"),
        col("r.event_date"),

        col("ride_summary"),

        col("silver_created_ts"),
        col("data_layer")
    )
)

print("✅ Ride Silver Transformation Completed")

# 📡 Trip Events Silver Transformation

In [0]:
# ==========================================
# Trip Events Silver Transformation
# ==========================================

trip_events_silver_df = (

    trip_events_bronze_df.alias("te")

    # Join Ride Information
    .join(

        rides_silver_df.select(

            "ride_id",

            "rider_id",
            "rider_name",

            "driver_id",
            "driver_name",

            "city_name",
            "zone_name",

            "ride_status"

        ).alias("r"),

        col("te.ride_id") == col("r.ride_id"),
        "left"
    )

    # Event Severity Classification
    .withColumn(
        "event_severity",
        when(
            col("event_type").isin(
                "RIDE_CANCELLED"
            ),
            "HIGH"
        )
        .when(
            col("event_type").isin(
                "PAYMENT_PROCESSED"
            ),
            "MEDIUM"
        )
        .otherwise("LOW")
    )

    # Event Category
    .withColumn(
        "event_category",
        when(
            col("event_type").isin(
                "RIDE_REQUESTED",
                "DRIVER_ASSIGNED",
                "DRIVER_ARRIVED"
            ),
            "PRE_RIDE"
        )
        .when(
            col("event_type").isin(
                "TRIP_STARTED",
                "TRIP_COMPLETED"
            ),
            "RIDE_EXECUTION"
        )
        .otherwise("POST_RIDE")
    )

    # Event Hour
    .withColumn(
        "event_hour",
        hour(col("event_timestamp"))
    )

    # AI Retrieval Summary
    .withColumn(
        "event_summary",
        concat_ws(
            " ",
            lit("Event"),
            col("event_type"),
            lit("occurred for ride"),
            col("te.ride_id"),
            lit("in city"),
            col("r.city_name"),
            lit("with ride status"),
            col("r.ride_status")
        )
    )

    # Metadata
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )

    # Final Columns
    .select(

        col("te.event_id"),
        col("te.ride_id"),

        col("r.rider_id"),
        col("r.rider_name"),

        col("r.driver_id"),
        col("r.driver_name"),

        col("te.event_type"),
        col("event_category"),
        col("event_severity"),

        col("te.event_timestamp"),
        col("event_hour"),

        col("r.city_name"),
        col("r.zone_name"),

        col("r.ride_status"),

        col("event_summary"),

        col("te.event_date"),

        col("silver_created_ts"),
        col("data_layer")
    )
)

print("✅ Trip Events Silver Transformation Completed")

# 🌍 City Silver Transformation

In [0]:
# ==========================================
# City Silver Transformation
# ==========================================

cities_silver_df = (

    cities_bronze_df

    # Standardize Columns
    .withColumn(
        "city_name",
        initcap(trim(col("city_name")))
    )

    .withColumn(
        "country",
        upper(trim(col("country")))
    )

    # Geographic Region Mapping
    .withColumn(
        "region",
        when(col("country") == "INDIA", "APAC")
        .when(col("country") == "USA", "NORTH_AMERICA")
        .otherwise("OTHER")
    )

    # Operational Tier
    .withColumn(
        "city_tier",
        when(
            col("city_name").isin(
                "Mumbai",
                "Delhi",
                "Bangalore",
                "New York"
            ),
            "TIER_1"
        )
        .otherwise("TIER_2")
    )

    # Metadata
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )
)

print("✅ City Silver Transformation Completed")

# 📍 Pricing Zone Silver Transformation

In [0]:
# ==========================================
# Pricing Zone Silver Transformation
# ==========================================

pricing_zones_silver_df = (

    pricing_zones_bronze_df.alias("pz")

    # Join City Information
    .join(
        cities_silver_df.alias("c"),
        col("pz.city_id") == col("c.city_id"),
        "left"
    )

    # Surge Category
    .withColumn(
        "surge_category",
        when(
            col("base_surge_multiplier") >= 4,
            "EXTREME"
        )
        .when(
            col("base_surge_multiplier") >= 3,
            "HIGH"
        )
        .when(
            col("base_surge_multiplier") >= 2,
            "MEDIUM"
        )
        .otherwise("LOW")
    )

    # AI Retrieval Summary
    .withColumn(
        "zone_summary",
        concat_ws(
            " ",
            lit("Pricing zone"),
            col("zone_name"),
            lit("located in"),
            col("c.city_name"),
            lit("with base surge multiplier"),
            col("base_surge_multiplier")
        )
    )

    # Metadata
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )

    # Final Columns
    .select(

        col("pz.zone_id"),
        col("pz.city_id"),

        col("c.city_name"),
        col("c.region"),

        col("pz.zone_name"),

        col("base_surge_multiplier"),
        col("surge_category"),

        col("zone_summary"),

        col("silver_created_ts"),
        col("data_layer")
    )
)

print("✅ Pricing Zone Silver Transformation Completed")

# 🧠 AI Semantic Operational Document Engineering

In [0]:
# ==========================================
# AI Semantic Operational Documents
# ==========================================

operational_documents_df = (

    rides_silver_df

    .withColumn(

        "document_text",

        concat_ws(

            " ",

            lit("Ride ID"),
            col("ride_id"),

            lit("occurred in city"),
            col("city_name"),

            lit("for rider"),
            col("rider_name"),

            lit("with driver"),
            col("driver_name"),

            lit("Vehicle type"),
            col("vehicle_type"),

            lit("Ride status"),
            col("ride_status"),

            lit("Fare amount"),
            col("fare_amount"),

            lit("Distance"),
            col("distance_km"),

            lit("Ride category"),
            col("ride_value_category"),

            lit("Ride time category"),
            col("ride_time_category"),

            lit("Payment method"),
            col("payment_method")
        )
    )

    # Document Metadata
    .withColumn(
        "document_type",
        lit("RIDE_OPERATION")
    )

    .withColumn(
        "semantic_domain",
        lit("RIDE_ANALYTICS")
    )

    # Retrieval Priority
    .withColumn(
        "retrieval_priority",
        when(
            col("ride_value_category") == "PREMIUM",
            1
        ).otherwise(2)
    )

    # Metadata
    .withColumn(
        "silver_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("SILVER")
    )

    # Final Columns
    .select(

        col("ride_id").alias("document_id"),

        col("document_type"),
        col("semantic_domain"),

        col("city_name"),
        col("zone_name"),

        col("ride_status"),

        col("document_text"),

        col("retrieval_priority"),

        col("event_date"),

        col("silver_created_ts"),

        col("data_layer")
    )
)

print("✅ AI Semantic Operational Documents Created")

# 🧹 Silver Layer Deduplication

In [0]:
# ==========================================
# Silver Layer Deduplication
# ==========================================

cities_silver_df = (
    cities_silver_df
    .dropDuplicates(["city_id"])
)

pricing_zones_silver_df = (
    pricing_zones_silver_df
    .dropDuplicates(["zone_id"])
)

drivers_silver_df = (
    drivers_silver_df
    .dropDuplicates(["driver_id"])
)

riders_silver_df = (
    riders_silver_df
    .dropDuplicates(["rider_id"])
)

rides_silver_df = (
    rides_silver_df
    .dropDuplicates(["ride_id"])
)

trip_events_silver_df = (
    trip_events_silver_df
    .dropDuplicates(["event_id"])
)

operational_documents_df = (
    operational_documents_df
    .dropDuplicates(["document_id"])
)

print("✅ All Silver Tables Deduplicated")

# 💾 Persist Silver Layer Tables

In [0]:
# ==========================================
# Persist Silver Layer Tables
# ==========================================

# ==========================================
# cities_silver
# ==========================================

cities_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        SILVER_TABLES["cities_silver"]
    )

print("✅ cities_silver persisted")

# ==========================================
# pricing_zones_silver
# ==========================================

pricing_zones_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        SILVER_TABLES["pricing_zones_silver"]
    )

print("✅ pricing_zones_silver persisted")

# ==========================================
# drivers_silver
# ==========================================

drivers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        SILVER_TABLES["drivers_silver"]
    )

print("✅ drivers_silver persisted")

# ==========================================
# riders_silver
# ==========================================

riders_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        SILVER_TABLES["riders_silver"]
    )

print("✅ riders_silver persisted")

# ==========================================
# rides_silver
# ==========================================

rides_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable(
        SILVER_TABLES["rides_silver"]
    )

print("✅ rides_silver persisted")

# ==========================================
# trip_events_silver
# ==========================================

trip_events_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable(
        SILVER_TABLES["trip_events_silver"]
    )

print("✅ trip_events_silver persisted")

# ==========================================
# operational_documents
# ==========================================

operational_documents_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable(
        SILVER_TABLES["operational_documents"]
    )

print("✅ operational_documents persisted")

print("✅ ALL SILVER TABLES PERSISTED SUCCESSFULLY")